# GLASS backend sweep explorer

Loads a `bench_mega_sweep` run (`bench/mega_sweep_*.txt`) and visualizes the
**warp vs block(SIMT) vs nvidia(MathDx)** ladder per op, plus the data-driven
backend winner per (op, N). This is the measurement behind `glass-defaults.cuh`'s
`suggested_backend<>()` — regenerate a per-host table with
`python bench/autotune.py --emit-defaults <sweep.txt>`.

Run from the `bench/` directory. Needs numpy + matplotlib.

In [ ]:
import re, glob
import numpy as np
import matplotlib.pyplot as plt

f = sorted(glob.glob("mega_sweep_*.txt"))[-1]   # most recent sweep
text = open(f).read()
print("sweep file:", f)

In [ ]:
# parse: (dtype, op, N) -> {block, warp, nvidia} best ns/problem at NPROB=8192
hdr = re.compile(r"NPROB=(\d+).*dtype=(f32|f64)")
row = re.compile(r"^(dot|gemv|gemm|chol|trsv|posv)\s+N=(\d+).*\|\|\s*block\s+tb\d+=([\d.]+)\s+warp\s+w\d+=([\d.]+)(?:\s+nv=([\d.]+))?")
data, dt, nprob = {}, None, None
for line in text.splitlines():
    if line.startswith("####"):
        m = hdr.search(line)
        if m: nprob, dt = int(m.group(1)), m.group(2)
        continue
    if nprob != 8192: continue          # the throughput regime
    m = row.match(line.strip())
    if m:
        op, N = m.group(1), int(m.group(2))
        d = {"block": float(m.group(3)), "warp": float(m.group(4))}
        if m.group(5): d["nvidia"] = float(m.group(5))
        data[(dt, op, N)] = d
OPS = ["dot","gemv","gemm","chol","trsv","posv"]
print("parsed", len(data), "cells")

## The ladder — ns/problem vs N, per backend (f32)

In [ ]:
def series(dt, op, key):
    pts = sorted((N, data[(dt,op,N)][key]) for (d,o,N) in data if d==dt and o==op and key in data[(dt,op,N)])
    return [p[0] for p in pts], [p[1] for p in pts]

fig, axes = plt.subplots(2, 3, figsize=(13, 7)); axes = axes.ravel()
for ax, op in zip(axes, OPS):
    for key, c in [("warp","tab:green"),("block","tab:blue"),("nvidia","tab:red")]:
        xs, ys = series("f32", op, key)
        if xs: ax.plot(xs, ys, "o-", color=c, label=key, ms=4)
    ax.set_title(f"{op} (f32, NPROB=8192)"); ax.set_xlabel("N"); ax.set_ylabel("ns/problem")
    ax.set_yscale("log"); ax.grid(alpha=.3); ax.legend(fontsize=8)
fig.tight_layout(); plt.show()

## Winner per (op, N) — what `suggested_backend` reflects

In [ ]:
for dt in ("f32", "f64"):
    Ns = sorted({N for (d,o,N) in data if d==dt})
    print(f"\n{dt} winner by op x N:")
    print("op     " + "".join(f"{N:>7}" for N in Ns))
    for op in OPS:
        cells = []
        for N in Ns:
            d = data.get((dt,op,N))
            cells.append(f"{min(d, key=d.get):>7}" if d else f"{'-':>7}")
        print(f"{op:6} " + "".join(cells))